# Demo 06 : Debugging with stream modes

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


## a · stream-modes

We run one graph in several stream modes and inspect state alongside a live model reply. A checkpoint saver enables checkpoint events.

In [2]:
from typing import TypedDict
from langgraph.graph import END, START, StateGraph
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.config import get_stream_writer

class DebugState(TypedDict):
    brief: str
    reply: str

def producer(state: DebugState):
    get_stream_writer()({"status": "reading the brief"})
    return {"reply": llm.invoke("You are the Producer at a game studio. Acknowledge this brief in one short line: " + state["brief"]).content}

builder = StateGraph(DebugState)
builder.add_node("producer", producer)
builder.add_edge(START, "producer")
builder.add_edge("producer", END)
graph = builder.compile(checkpointer=InMemorySaver())
inputs = {"brief": "Give Comet Snake a shield, one second chance against a comet.", "reply": ""}
config = {"configurable": {"thread_id": "debug-live"}}
modes = ["values", "updates", "debug", "messages", "custom", "checkpoints", "tasks"]
parts = list(graph.stream(inputs, config, stream_mode=modes, version="v2"))
for part in parts:
    if part["type"] == "messages":
        continue
    if part["type"] in {"debug", "tasks", "checkpoints"}:
        print(part["type"], ":", str(part["data"])[:500])
    else:
        print(part["type"], ":", part["data"])
tokens = [part["data"][0].content for part in parts if part["type"] == "messages"]
print("message chunks :", len(tokens))
print("reply :", graph.get_state(config).values["reply"])
first_values = next(part for part in parts if part["type"] == "values")
print("values part keys :", sorted(first_values))
print("namespace :", parts[0]["ns"])
print("emitted modes :", sorted({part["type"] for part in parts}))

checkpoints : {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'debug-live', 'checkpoint_id': '1f1d4b1e-ac1a-6c48-bfff-21ab0244a366'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'f6de100a-2edc-2983-f115-6869dac499ca', 'name': '__start__', 'interrupts': (), 'state': None}]}
debug : {'step': -1, 'timestamp': '2026-10-30T22:33:23.806700+00:00', 'type': 'checkpoint', 'payload': {'config': {'configurable': {'checkpoint_ns': '', 'thread_id': 'debug-live', 'checkpoint_id': '1f1d4b1e-ac1a-6c48-bfff-21ab0244a366'}}, 'parent_config': None, 'values': {}, 'metadata': {'source': 'input', 'step': -1, 'parents': {}}, 'next': ['__start__'], 'tasks': [{'id': 'f6de100a-2edc-2983-f115-6869dac499ca', 'name': '__start__', 'interrupts': (), 'state': None}]}}
values : {'brief': 'Give Comet Snake a shield, one second chance against a comet.', 'reply': ''}
checkpoints : {'config': {'configurable': {'che

We inspect experimental v3 projections on a deterministic graph to avoid another model call. Iterating a projection drives the run ; output becomes available when the run finishes.

In [3]:
class Attempts(TypedDict):
    attempts: int

small = StateGraph(Attempts)
small.add_node("count", lambda state: {"attempts": state["attempts"] + 1})
small.add_edge(START, "count")
small.add_edge("count", END)
small_graph = small.compile()
run = small_graph.stream_events({"attempts": 0}, version="v3")
for state in run.values:
    print("v3 value :", state)
print("v3 output :", run.output)
print("v3 interrupted :", run.interrupted)
parent = StateGraph(Attempts)
parent.add_node("counter", small_graph)
parent.add_edge(START, "counter")
parent.add_edge("counter", END)
lifecycle = parent.compile().stream_events({"attempts": 0}, version="v3")
for event in lifecycle.lifecycle:
    print("v3 lifecycle :", event)

v3 value : {'attempts': 0}
v3 value : {'attempts': 1}
v3 output : {'attempts': 1}
v3 interrupted : False
v3 lifecycle : {'event': 'started', 'namespace': ['counter:db71af90-9a9b-9247-da0a-0616110352b2'], 'graph_name': 'counter', 'trigger_call_id': 'db71af90-9a9b-9247-da0a-0616110352b2'}
v3 lifecycle : {'event': 'completed', 'namespace': ['counter:db71af90-9a9b-9247-da0a-0616110352b2']}


## Things to try

Optional : we can change a brief, a thread id or a returned update and compare the saved state.